In [1]:
from torch.nn.modules.loss import CrossEntropyLoss
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

#dataset and dataloader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])

trainset = CIFAR10(root='/content/drive/MyDrive/Dataset',train=True,download=True,transform=transform)
testset = CIFAR10(root='/content/drive/MyDrive/Dataset',train=False,download=True,transform=transform)
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(testset,batch_size=64,shuffle=False)

#building model
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()
    self.conv_layers = nn.Sequential(

        #1ST LAYER
        nn.Conv2d(3,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),
        #2nd layer
        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),
        #3rd layer
        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),
    )

    self.fc_layers = nn.Sequential(
        nn.Linear(4*4*128,256),
        nn.ReLU(),
        nn.Linear(256,10)
    )
  def forward(self,x):
    x = self.conv_layers(x)
    x = x.view(x.size(0),-1)
    x = self.fc_layers(x)
    return x

model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

#train
epochs = 10
for epoch in range(epochs):
  epoch_training_loss = 0.0
  model.train()
  for images,labels in trainloader:
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs,labels)
    loss.backward()
    optimizer.step()
    epoch_training_loss += loss.item()
  print(f"epoch {epoch+1}/{epochs} & loss {epoch_training_loss/len(trainloader)}")

#evaluation
correct_labels = 0
total_labels = 0
model.eval()
with torch.no_grad():
  for images,labels in testloader:
    outputs = model(images)
    _,predicted = torch.max(outputs,1)
    correct_labels += (predicted == labels).sum().item()
    total_labels += labels.size(0)
print(f"accuracy = {(correct_labels/total_labels)*100}")

epoch 1/10 & loss 1.3672086403650396
epoch 2/10 & loss 0.9302345757441752
epoch 3/10 & loss 0.7417004507063599
epoch 4/10 & loss 0.6161690038793227
epoch 5/10 & loss 0.5158844939278214
epoch 6/10 & loss 0.4193153133058487
epoch 7/10 & loss 0.32828413273977197
epoch 8/10 & loss 0.252715688336955
epoch 9/10 & loss 0.19227130029379103
epoch 10/10 & loss 0.14745295441011563
accuracy = 75.44
